In [ ]:
!pip install ultralytics pytorch-grad-cam pandas -q

import os
import random
import shutil
import json
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

from torchvision import datasets, transforms
from torchvision.models import resnet18, ResNet18_Weights, efficientnet_b0, EfficientNet_B0_Weights

from ultralytics import YOLO
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    precision_recall_fscore_support, accuracy_score, f1_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.utils.class_weight import compute_class_weight
from pathlib import Path

try:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    GRADCAM_AVAILABLE = True
except ImportError:
    GRADCAM_AVAILABLE = False
    print("pytorch-grad-cam kurulamadı — Grad-CAM hücresi atlanabilir.")


In [2]:

base_path = "/kaggle/input/datasets/esmanurozcelik/vhrships-dataset/VHR dataset/labeledGT"

all_images = [f for f in os.listdir(base_path) if f.endswith(".jpg")]

random.shuffle(all_images)

all_images = all_images[:400]   

print("Kullanılan veri:", len(all_images))

Kullanılan veri: 400


In [3]:
os.makedirs("/kaggle/working/dataset/images/train", exist_ok=True)
os.makedirs("/kaggle/working/dataset/labels/train", exist_ok=True)

In [4]:
valid_images = []

for i, img_name in enumerate(all_images):

    if i % 50 == 0:
        print(f"{i}/{len(all_images)}")

    img_path = os.path.join(base_path, img_name)
    img = cv2.imread(img_path)

    if img is None:
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 200, 255, cv2.THRESH_BINARY_INV)

    contours, _ = cv2.findContours(
        thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    if len(contours) == 0:
        continue

    biggest = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(biggest)


    area = w * h
    aspect_ratio = max(w, h) / min(w, h) if min(w, h) > 0 else 1
    
    is_tanker = area > 8000 and aspect_ratio > 2.2
    
    is_horizontal = w > h
    
    # Klas numarasını seçiyoruz (0, 1, 2 veya 3)
    if not is_tanker and is_horizontal:
        class_id = 0  # Horizontal_Cargo
    elif not is_tanker and not is_horizontal:
        class_id = 1  # Vertical_Cargo
    elif is_tanker and is_horizontal:
        class_id = 2  # Horizontal_Tanker
    else:
        class_id = 3  # Vertical_Tanker
    # ----------------------------------------------------------------

    h_img, w_img = img.shape[:2]

    x_center = (x + w / 2) / w_img
    y_center = (y + h / 2) / h_img
    w_norm = w / w_img
    h_norm = h / h_img


    label = f"{class_id} {x_center} {y_center} {w_norm} {h_norm}"

    cv2.imwrite(f"/kaggle/working/dataset/images/train/{img_name}", img)

    with open(
        f"/kaggle/working/dataset/labels/train/{img_name.replace('.jpg','.txt')}",
        "w"
    ) as f:
        f.write(label)

    valid_images.append(img_name)

print("Kullanılabilir veri hazırlığı bitti:", len(valid_images))

0/400
50/400
100/400
150/400
200/400
250/400
300/400
350/400
Kullanılabilir veri hazırlığı bitti: 400


In [5]:
img_folder = "/kaggle/working/dataset/images/train"
label_folder = "/kaggle/working/dataset/labels/train"

images = os.listdir(img_folder)
random.shuffle(images)

split = int(len(images) * 0.8)

val_images = images[split:]

os.makedirs("/kaggle/working/dataset/images/val", exist_ok=True)
os.makedirs("/kaggle/working/dataset/labels/val", exist_ok=True)

for img in val_images:

    shutil.move(
        os.path.join(img_folder, img),
        os.path.join("/kaggle/working/dataset/images/val", img)
    )

    txt = img.replace(".jpg", ".txt")

    shutil.move(
        os.path.join(label_folder, txt),
        os.path.join("/kaggle/working/dataset/labels/val", txt)
    )

In [6]:
with open("/kaggle/working/dataset.yaml", "w") as f:
    f.write("""
path: /kaggle/working/dataset

train: images/train
val: images/val

names:
  0: Horizontal_Cargo
  1: Vertical_Cargo
  2: Horizontal_Tanker
  3: Vertical_Tanker
""")

In [ ]:
from ultralytics import YOLO

# YOLO11 — gemi tespiti (nesne tespiti aşaması)
yolo_model = YOLO("yolo11n.pt")
model = yolo_model  # mevcut eğitim hücreleriyle uyumluluk


In [ ]:

# YOLO11 — optimize edilmiş hiperparametreler (VHR uydu görüntüleri)
# Öneri gerekçesi:
#   epochs=50  : pseudo-label veri setinde yeterli yakınsama
#   imgsz=640  : VHR çözünürlüğü / GPU bellek dengesi
#   batch=16   : stabil gradient; bellek yetersizse 8'e düşürün
#   lr0=0.001  : pretrained ağırlıklarda güvenli fine-tune
#   patience=15: erken durdurma
#   cos_lr=True: cosine learning rate schedule

YOLO_EPOCHS = 50
YOLO_IMGSZ = 640
YOLO_BATCH = 16
YOLO_LR0 = 0.001
YOLO_PATIENCE = 15

device_yolo = "0" if torch.cuda.is_available() else "cpu"
print(f"YOLO11 eğitimi {device_yolo} üzerinde başlatılıyor...")
print(f"  epochs={YOLO_EPOCHS}, imgsz={YOLO_IMGSZ}, batch={YOLO_BATCH}, lr0={YOLO_LR0}")

yolo_train_results = yolo_model.train(
    data="/kaggle/working/dataset.yaml",
    epochs=YOLO_EPOCHS,
    imgsz=YOLO_IMGSZ,
    batch=YOLO_BATCH,
    device=device_yolo,
    lr0=YOLO_LR0,
    lrf=0.01,
    cos_lr=True,
    patience=YOLO_PATIENCE,
    augment=True,
    mosaic=0.8,
    mixup=0.1,
    plots=True,
    save=True,
    verbose=True,
)
model = yolo_model
print("YOLO11 eğitimi tamamlandı.")


In [9]:
model = YOLO("/kaggle/working/runs/detect/train/weights/best.pt")

os.makedirs("/kaggle/working/crops", exist_ok=True)

for i, img_name in enumerate(valid_images[:150]):  

    if i % 20 == 0:
        print(f"Crop {i}")

    img_path = os.path.join(base_path, img_name) 
    img = cv2.imread(img_path)

    results = model(img_path)

    if len(results[0].boxes) == 0:
        continue

    for j, box in enumerate(results[0].boxes.xyxy):

        x1, y1, x2, y2 = map(int, box)

       
        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(img.shape[1], x2)
        y2 = min(img.shape[0], y2)

        crop = img[y1:y2, x1:x2]

        if crop.size == 0:
            continue

        cv2.imwrite(f"/kaggle/working/crops/{img_name}_{j}.jpg", crop)

Crop 0

image 1/1 /kaggle/input/datasets/esmanurozcelik/vhrships-dataset/VHR dataset/labeledGT/PE_008.jpg: 384x640 1 Horizontal_Cargo, 165.5ms
Speed: 8.8ms preprocess, 165.5ms inference, 6.1ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 /kaggle/input/datasets/esmanurozcelik/vhrships-dataset/VHR dataset/labeledGT/SA_619.jpg: 384x640 1 Horizontal_Cargo, 101.9ms
Speed: 3.2ms preprocess, 101.9ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 /kaggle/input/datasets/esmanurozcelik/vhrships-dataset/VHR dataset/labeledGT/PE_1695.jpg: 384x640 1 Horizontal_Cargo, 101.2ms
Speed: 3.5ms preprocess, 101.2ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 /kaggle/input/datasets/esmanurozcelik/vhrships-dataset/VHR dataset/labeledGT/bck_400.jpg: 384x640 1 Horizontal_Cargo, 102.7ms
Speed: 2.8ms preprocess, 102.7ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 /kaggle/input/datasets/esmanurozcelik/vhrshi

In [10]:
model = YOLO("/kaggle/working/runs/detect/train/weights/best.pt")

In [11]:
test_images = valid_images[:5]  

In [ ]:
# =============================================================================
# Gelişmiş Karar Destek Sistemi (Hybrid Scoring DSS)
# Çoklu özellik + skor tabanlı hibrit karar mekanizması
# =============================================================================

tanker_database = {
    "Aframax": {
        "ship_type": "Crude Oil Tanker (Aframax)",
        "cargo_capacity": "700,000 – 850,000 barrels",
        "dwt_range": "80,000 – 120,000 DWT",
        "environmental_risk": "Medium — regional spill potential",
        "strategic_importance": "Regional crude oil corridors",
    },
    "Suezmax": {
        "ship_type": "Crude Oil Tanker (Suezmax)",
        "cargo_capacity": "1,000,000 – 1,200,000 barrels",
        "dwt_range": "120,000 – 200,000 DWT",
        "environmental_risk": "High — large-volume cargo",
        "strategic_importance": "Suez Canal / Mediterranean routes",
    },
    "VLCC": {
        "ship_type": "Very Large Crude Carrier (VLCC)",
        "cargo_capacity": "2,000,000 – 2,500,000 barrels",
        "dwt_range": "200,000 – 320,000 DWT",
        "environmental_risk": "Very High — catastrophic spill risk",
        "strategic_importance": "Global crude supply chains",
    },
    "ULCC": {
        "ship_type": "Ultra Large Crude Carrier (ULCC)",
        "cargo_capacity": "3,000,000+ barrels",
        "dwt_range": "320,000+ DWT",
        "environmental_risk": "Critical — maximum cargo volume",
        "strategic_importance": "Strategic chokepoints and mega-hub ports",
    },
}

OTHER_SHIP_PROFILE = {
    "ship_type": "Commercial Vessel (Non-Tanker)",
    "cargo_capacity": "N/A",
    "dwt_range": "N/A",
    "environmental_risk": "Low — non-petroleum cargo",
    "strategic_importance": "General maritime traffic",
}

# Kategori profilleri — normalize alan oranı + en-boy oranı aralıkları
CATEGORY_PROFILES = {
    "Aframax": {"area_norm": (0.003, 0.025), "aspect": (1.5, 3.2), "env_base": 22, "strategic_base": 30},
    "Suezmax": {"area_norm": (0.015, 0.06),  "aspect": (2.0, 4.5), "env_base": 42, "strategic_base": 52},
    "VLCC":    {"area_norm": (0.04, 0.14),   "aspect": (2.5, 5.5), "env_base": 68, "strategic_base": 75},
    "ULCC":    {"area_norm": (0.10, 0.35),   "aspect": (3.0, 7.0), "env_base": 88, "strategic_base": 92},
}

CATEGORY_ORDER = ["Aframax", "Suezmax", "VLCC", "ULCC"]


def estimate_tanker_category(width, height):
    """Geriye dönük uyumluluk — kural tabanlı fallback."""
    area = width * height
    if area < 3000:
        return "Aframax"
    elif area < 15000:
        return "Suezmax"
    elif area < 40000:
        return "VLCC"
    return "ULCC"


def estimate_direction(width, height):
    if width > height:
        return "East-West Route"
    return "North-South Route"


def _range_membership(value, low, high):
    """Değer aralık içindeyse 1.0; dışında mesafeye göre azalır."""
    if low <= value <= high:
        return 1.0
    mid = (low + high) / 2
    half = (high - low) / 2 + 1e-6
    dist = min(abs(value - low), abs(value - high))
    return float(max(0.0, 1.0 - dist / half))


def extract_ship_features(bbox_w, bbox_h, img_w, img_h, yolo_conf, fuel_prob):
    area = bbox_w * bbox_h
    img_area = max(img_w * img_h, 1)
    aspect = max(bbox_w, bbox_h) / max(min(bbox_w, bbox_h), 1)
    return {
        "bbox_area": area,
        "area_norm": area / img_area,
        "aspect_ratio": aspect,
        "yolo_confidence": float(yolo_conf),
        "fuel_probability": float(fuel_prob),
        "image_resolution": img_w * img_h,
        "image_width": img_w,
        "image_height": img_h,
        "ship_direction": estimate_direction(bbox_w, bbox_h),
        "is_horizontal": bbox_w > bbox_h,
    }


def compute_category_scores(features):
    """Hibrit skor: profil uyumu (70%) + güven bileşeni (30%)."""
    raw = {}
    for cat in CATEGORY_ORDER:
        prof = CATEGORY_PROFILES[cat]
        s_area = _range_membership(features["area_norm"], *prof["area_norm"])
        s_aspect = _range_membership(features["aspect_ratio"], *prof["aspect"])
        profile_score = 0.55 * s_area + 0.45 * s_aspect
        conf_score = 0.4 * features["yolo_confidence"] + 0.6 * features["fuel_probability"]
        raw[cat] = 0.70 * profile_score + 0.30 * conf_score

    arr = np.array([raw[c] for c in CATEGORY_ORDER], dtype=np.float64)
    arr = np.clip(arr, 1e-6, None)
    probs = arr / arr.sum()
    best_idx = int(np.argmax(probs))
    return {c: float(probs[i]) for i, c in enumerate(CATEGORY_ORDER)}, CATEGORY_ORDER[best_idx], float(probs[best_idx])


def environmental_risk_score(category, features):
    base = CATEGORY_PROFILES[category]["env_base"]
    area_boost = min(12.0, features["area_norm"] * 400)
    conf_boost = features["fuel_probability"] * 8
    score = min(100.0, base + area_boost + conf_boost)
    return round(score, 1)


def strategic_importance_score(category, features):
    base = CATEGORY_PROFILES[category]["strategic_base"]
    route_boost = 6.0 if features["is_horizontal"] else 3.0
    yolo_boost = features["yolo_confidence"] * 10
    score = min(100.0, base + route_boost + yolo_boost)
    return round(score, 1)


def overall_risk_score(env_score, strategic_score, category_conf):
    combined = 0.60 * env_score + 0.30 * strategic_score + 0.10 * (category_conf * 100)
    return round(min(100.0, combined), 1)


def risk_level(score):
    if score <= 25:
        return "Düşük"
    if score <= 50:
        return "Orta"
    if score <= 75:
        return "Yüksek"
    return "Kritik"


def threat_level_label(env_score):
    if env_score <= 25:
        return "Düşük Tehdit (0–25)"
    if env_score <= 50:
        return "Orta Tehdit (26–50)"
    if env_score <= 75:
        return "Yüksek Tehdit (51–75)"
    return "Kritik Tehdit (76–100)"


def hybrid_confidence_score(features, category_conf, category_probs):
    entropy = -sum(p * np.log(p + 1e-9) for p in category_probs.values())
    max_entropy = np.log(len(CATEGORY_ORDER))
    ambiguity = entropy / max_entropy if max_entropy > 0 else 0
    score = (
        0.25 * features["yolo_confidence"]
        + 0.35 * features["fuel_probability"]
        + 0.25 * category_conf
        + 0.15 * (1.0 - ambiguity)
    ) * 100
    return round(min(100.0, max(0.0, score)), 1)


def build_decision_support_report(fuel_label, bbox_w, bbox_h, img_w, img_h,
                                  yolo_conf, fuel_prob, other_prob=None):
    is_fuel = fuel_label.lower() == "fuel"
    report = {
        "Ship Type": OTHER_SHIP_PROFILE["ship_type"],
        "Fuel / Other": fuel_label.capitalize(),
        "Tanker Category": "N/A",
        "Estimated Route": "N/A",
        "Risk Level": "Düşük",
        "Risk Score": 0.0,
        "Environmental Risk Score": 0.0,
        "Environmental Threat Level": "Düşük Tehdit (0–25)",
        "Strategic Importance Score": 0.0,
        "Confidence Score": round(float(yolo_conf) * 100, 1),
        "Cargo Capacity": OTHER_SHIP_PROFILE["cargo_capacity"],
        "DWT Range": OTHER_SHIP_PROFILE["dwt_range"],
        "Environmental Risk": OTHER_SHIP_PROFILE["environmental_risk"],
        "Strategic Importance": OTHER_SHIP_PROFILE["strategic_importance"],
        "Category Probabilities": {},
        "Features": {},
    }

    if not is_fuel:
        return report

    features = extract_ship_features(bbox_w, bbox_h, img_w, img_h, yolo_conf, fuel_prob)
    cat_probs, best_cat, cat_conf = compute_category_scores(features)
    env_score = environmental_risk_score(best_cat, features)
    strat_score = strategic_importance_score(best_cat, features)
    risk_score = overall_risk_score(env_score, strat_score, cat_conf)
    conf_score = hybrid_confidence_score(features, cat_conf, cat_probs)
    profile = tanker_database[best_cat]

    report.update({
        "Ship Type": profile["ship_type"],
        "Tanker Category": best_cat,
        "Estimated Route": features["ship_direction"],
        "Risk Level": risk_level(risk_score),
        "Risk Score": risk_score,
        "Environmental Risk Score": env_score,
        "Environmental Threat Level": threat_level_label(env_score),
        "Strategic Importance Score": strat_score,
        "Confidence Score": conf_score,
        "Cargo Capacity": profile["cargo_capacity"],
        "DWT Range": profile["dwt_range"],
        "Environmental Risk": profile["environmental_risk"],
        "Strategic Importance": profile["strategic_importance"],
        "Category Probabilities": {k: round(v, 4) for k, v in cat_probs.items()},
        "Features": {k: (round(v, 4) if isinstance(v, float) else v) for k, v in features.items()},
    })
    return report


def generate_fuel_analysis_report(report, ship_id=1):
    """Fuel gemileri için otomatik metin raporu."""
    lines = [
        "=" * 68,
        f"  YAKIT TANKERİ ANALİZ RAPORU — Gemi #{ship_id}",
        "=" * 68,
        f"  Gemi Tipi                  : {report['Ship Type']}",
        f"  Sınıflandırma              : {report['Fuel / Other']}",
        f"  Tanker Kategorisi          : {report['Tanker Category']}",
        f"  Tahmini Rota               : {report['Estimated Route']}",
        f"  Risk Puanı                 : {report['Risk Score']}/100 ({report['Risk Level']})",
        f"  Çevresel Risk Skoru        : {report['Environmental Risk Score']}/100",
        f"  Çevresel Tehdit Seviyesi   : {report['Environmental Threat Level']}",
        f"  Stratejik Önem Puanı       : {report['Strategic Importance Score']}/100",
        f"  Güven Skoru                : {report['Confidence Score']}/100",
        f"  Kargo Kapasitesi           : {report['Cargo Capacity']}",
        f"  DWT Aralığı                : {report['DWT Range']}",
    ]
    if report.get("Category Probabilities"):
        lines.append("  Kategori Olasılıkları      :")
        for cat, p in report["Category Probabilities"].items():
            lines.append(f"    - {cat:<10}: {p*100:.1f}%")
    if report.get("Features"):
        f = report["Features"]
        lines += [
            f"  BBox Alanı (norm.)         : {f.get('area_norm', 'N/A')}",
            f"  En-Boy Oranı               : {f.get('aspect_ratio', 'N/A')}",
            f"  YOLO Güveni                : {f.get('yolo_confidence', 'N/A')}",
            f"  Fuel Olasılığı             : {f.get('fuel_probability', 'N/A')}",
        ]
    lines.append("=" * 68)
    return "\n".join(lines)


def print_decision_support_report(report, ship_id=1):
    print(generate_fuel_analysis_report(report, ship_id))


print("Gelişmiş Hibrit Karar Destek Sistemi yüklendi.")
print(f"  Kategoriler: {CATEGORY_ORDER}")
print("  Karar mekanizması: profil uyumu + YOLO/Fuel güven skoru (hibrit)")


In [15]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print(root)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/esmanurozcelik
/kaggle/input/datasets/esmanurozcelik/vhrships-dataset
/kaggle/input/datasets/esmanurozcelik/vhrships-dataset/VHR dataset
/kaggle/input/datasets/esmanurozcelik/vhrships-dataset/VHR dataset/_mergedData
/kaggle/input/datasets/esmanurozcelik/vhrships-dataset/VHR dataset/labeledGT


In [ ]:
# Fuel vs Other — Veri seti, split ve DataLoader
from pathlib import Path
from PIL import Image

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT_ROOT = Path(".")
DATASET_ROOT = next(
    (p for p in PROJECT_ROOT.iterdir() if p.is_dir() and (p / "fuel").exists() and (p / "other").exists()),
    None,
)
if DATASET_ROOT is None:
    raise FileNotFoundError("crops_azaltılmış klasörü bulunamadı (fuel/ ve other/ alt klasörleri gerekli)")

print(f"Veri seti: {DATASET_ROOT.resolve()}")
CLASS_NAMES = ["fuel", "other"]

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


class ImageFolderSubset(Dataset):
    def __init__(self, imagefolder, indices, transform):
        self.imagefolder = imagefolder
        self.indices = list(indices)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        path, target = self.imagefolder.samples[self.indices[idx]]
        image = Image.open(path).convert("RGB")
        return self.transform(image), target

    def get_path(self, idx):
        return self.imagefolder.samples[self.indices[idx]][0]


base_dataset = datasets.ImageFolder(DATASET_ROOT, transform=None)
targets = np.array(base_dataset.targets)
all_indices = np.arange(len(base_dataset))

idx_trainval, idx_test = train_test_split(
    all_indices, test_size=0.10, stratify=targets, random_state=SEED
)
trainval_targets = targets[idx_trainval]
idx_train, idx_val = train_test_split(
    idx_trainval, test_size=0.111, stratify=trainval_targets, random_state=SEED
)  # ~80/10/10

test_dataset = ImageFolderSubset(base_dataset, idx_test, eval_transform)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=0)

print(f"Toplam: {len(base_dataset)} | Train-val pool: {len(idx_trainval)} | Test: {len(idx_test)}")
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    print(f"  {cls_name}: total={(targets==cls_idx).sum()}, test={(targets[idx_test]==cls_idx).sum()}")

K_FOLDS = 5
BATCH_SIZE = 16
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Cihaz: {device} | K-Fold: {K_FOLDS}")


In [ ]:
# Model fabrikası ve eğitim yardımcıları

def create_model(architecture="resnet18", num_classes=2):
    if architecture == "resnet18":
        model = resnet18(weights=ResNet18_Weights.DEFAULT)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        cam_layer = "layer4"
    elif architecture == "efficientnet_b0":
        model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
        cam_layer = "features"
    else:
        raise ValueError(f"Bilinmeyen mimari: {architecture}")
    return model, cam_layer


def get_cam_target_layer(model, architecture):
    if architecture == "resnet18":
        return [model.layer4[-1]]
    return [model.features[-1]]


def evaluate_model(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="macro", zero_division=0
    )
    acc = accuracy_score(all_labels, all_preds)
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "preds": all_preds,
        "labels": all_labels,
    }


def train_one_model(model, train_loader, val_loader, device, epochs=25, patience=6, lr=1e-4):
    train_labels_local = []
    for _, lbl in train_loader.dataset:
        train_labels_local.append(lbl)
    cw = compute_class_weight("balanced", classes=np.array([0, 1]), y=train_labels_local)
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float32).to(device))
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

    best_f1, best_state, wait = 0.0, None, 0
    for epoch in range(1, epochs + 1):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()

        val_metrics = evaluate_model(model, val_loader, device)
        scheduler.step(val_metrics["f1"])
        if val_metrics["f1"] > best_f1:
            best_f1 = val_metrics["f1"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    if best_state:
        model.load_state_dict(best_state)
    return model, best_f1


print("Eğitim yardımcıları hazır: resnet18, efficientnet_b0")


In [ ]:
# K-Fold Cross Validation — ResNet18 vs EfficientNet-B0
EPOCHS_FOLD = 25
PATIENCE_FOLD = 6
ARCHITECTURES = ["resnet18", "efficientnet_b0"]

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
fold_results = {arch: [] for arch in ARCHITECTURES}
fold_models = {}

for arch in ARCHITECTURES:
    print(f"\n{'='*60}\n  {arch.upper()} — {K_FOLDS}-Fold CV\n{'='*60}")
    for fold, (tr_idx, va_idx) in enumerate(skf.split(idx_trainval, trainval_targets), start=1):
        train_ds = ImageFolderSubset(base_dataset, idx_trainval[tr_idx], train_transform)
        val_ds = ImageFolderSubset(base_dataset, idx_trainval[va_idx], eval_transform)
        train_ldr = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
        val_ldr = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

        model_fold, cam_layer_name = create_model(arch)
        model_fold = model_fold.to(device)
        model_fold, best_f1 = train_one_model(
            model_fold, train_ldr, val_ldr, device,
            epochs=EPOCHS_FOLD, patience=PATIENCE_FOLD,
        )
        metrics = evaluate_model(model_fold, val_ldr, device)
        metrics["fold"] = fold
        fold_results[arch].append(metrics)
        print(
            f"  Fold {fold}: Acc={metrics['accuracy']:.4f} "
            f"P={metrics['precision']:.4f} R={metrics['recall']:.4f} F1={metrics['f1']:.4f}"
        )
        if metrics["f1"] >= max(m["f1"] for m in fold_results[arch]):
            fold_models[arch] = model_fold

# Ortalama performans tablosu
rows = []
for arch in ARCHITECTURES:
    mlist = fold_results[arch]
    rows.append({
        "Model": arch,
        "Accuracy": np.mean([m["accuracy"] for m in mlist]),
        "Precision": np.mean([m["precision"] for m in mlist]),
        "Recall": np.mean([m["recall"] for m in mlist]),
        "F1": np.mean([m["f1"] for m in mlist]),
        "Std_F1": np.std([m["f1"] for m in mlist]),
    })

cv_comparison_df = pd.DataFrame(rows)
cv_comparison_df = cv_comparison_df.round(4)
print("\n=== K-Fold Ortalama Performans (Karşılaştırmalı Tablo) ===")
display(cv_comparison_df)
cv_comparison_df.to_csv("kfold_comparison.csv", index=False)


In [ ]:
# Hold-out test seti değerlendirmesi ve en iyi model seçimi
weights_dir = Path("weights")
weights_dir.mkdir(exist_ok=True)

test_rows = []
best_arch = cv_comparison_df.loc[cv_comparison_df["F1"].idxmax(), "Model"]
print(f"K-Fold sonucuna göre en iyi mimari: {best_arch}")

final_models = {}
for arch in ARCHITECTURES:
    model_final = fold_models[arch]
    test_metrics = evaluate_model(model_final, test_loader, device)
    test_rows.append({
        "Model": arch,
        "Accuracy": test_metrics["accuracy"],
        "Precision": test_metrics["precision"],
        "Recall": test_metrics["recall"],
        "F1": test_metrics["f1"],
    })
    path = weights_dir / f"fuel_other_{arch}_best.pth"
    torch.save(model_final.state_dict(), path)
    final_models[arch] = model_final
    print(f"{arch}: Test Acc={test_metrics['accuracy']:.4f} F1={test_metrics['f1']:.4f} -> {path}")

test_comparison_df = pd.DataFrame(test_rows).round(4)
print("\n=== Test Seti Karşılaştırmalı Tablo ===")
display(test_comparison_df)
test_comparison_df.to_csv("test_comparison.csv", index=False)

# Confusion matrix — en iyi model
best_model = final_models[best_arch]
best_metrics = evaluate_model(best_model, test_loader, device)
cm = confusion_matrix(best_metrics["labels"], best_metrics["preds"], labels=[0, 1])
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Test CM — {best_arch}")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

print(classification_report(best_metrics["labels"], best_metrics["preds"], target_names=CLASS_NAMES))
BEST_ARCHITECTURE = best_arch
best_model_path = weights_dir / f"fuel_other_{best_arch}_best.pth"


In [ ]:
# Grad-CAM — modelin karar verirken odaklandığı bölgeler
if not GRADCAM_AVAILABLE:
    print("Grad-CAM atlandı: pytorch-grad-cam kurulu değil.")
else:
    gradcam_model = final_models[BEST_ARCHITECTURE]
    gradcam_model.eval()
    target_layers = get_cam_target_layer(gradcam_model, BEST_ARCHITECTURE)
    cam = GradCAM(model=gradcam_model, target_layers=target_layers)

    n_samples = min(4, len(test_dataset))
    fig, axes = plt.subplots(2, n_samples, figsize=(4 * n_samples, 8))
    if n_samples == 1:
        axes = np.array([[axes[0]], [axes[1]]])

    for i in range(n_samples):
        img_tensor, label = test_dataset[i]
        input_tensor = img_tensor.unsqueeze(0).to(device)
        targets = [ClassifierOutputTarget(label)]
        grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0]

        rgb = img_tensor.permute(1, 2, 0).numpy()
        rgb = rgb * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        rgb = np.clip(rgb, 0, 1)
        cam_image = cv2.applyColorMap(np.uint8(255 * grayscale_cam), cv2.COLORMAP_JET)
        cam_image = cv2.cvtColor(cam_image, cv2.COLOR_BGR2RGB)
        overlay = np.clip(0.55 * rgb + 0.45 * (cam_image / 255.0), 0, 1)

        axes[0, i].imshow(rgb)
        axes[0, i].set_title(f"GT: {CLASS_NAMES[label]}")
        axes[0, i].axis("off")
        axes[1, i].imshow(overlay)
        axes[1, i].set_title(f"Grad-CAM ({BEST_ARCHITECTURE})")
        axes[1, i].axis("off")

    plt.suptitle("Grad-CAM Analizi — Sınıflandırma Odak Haritası", fontsize=13)
    plt.tight_layout()
    plt.savefig("gradcam_analysis.png", dpi=150)
    plt.show()
    print("Grad-CAM görseli kaydedildi: gradcam_analysis.png")


In [ ]:
# Olası etiket hatası tespiti
def detect_label_errors(model, dataset, device, confidence_threshold=0.85):
    """Yüksek güvenle yanlış tahmin edilen örnekleri şüpheli etiket olarak işaretle."""
    model.eval()
    suspicious = []
    for i in range(len(dataset)):
        img, true_label = dataset[i]
        path = dataset.get_path(i)
        with torch.no_grad():
            logits = model(img.unsqueeze(0).to(device))
            probs = torch.softmax(logits, dim=1)[0]
            pred_label = logits.argmax(dim=1).item()
            conf = probs[pred_label].item()
            fuel_prob = probs[0].item()

        if pred_label != true_label and conf >= confidence_threshold:
            suspicious.append({
                "path": path,
                "true_label": CLASS_NAMES[true_label],
                "pred_label": CLASS_NAMES[pred_label],
                "confidence": round(conf, 4),
                "fuel_prob": round(fuel_prob, 4),
            })
    return suspicious


# Tüm veri üzerinde tarama (train+val+test pool)
full_eval_ds = ImageFolderSubset(base_dataset, all_indices, eval_transform)
label_scan_model = final_models[BEST_ARCHITECTURE]
suspicious_labels = detect_label_errors(label_scan_model, full_eval_ds, device, confidence_threshold=0.80)

print(f"Taranan örnek: {len(full_eval_ds)}")
print(f"Şüpheli etiket sayısı: {len(suspicious_labels)}")
if suspicious_labels:
    suspect_df = pd.DataFrame(suspicious_labels)
    display(suspect_df.head(20))
    suspect_df.to_csv("suspicious_labels.csv", index=False)
    print("Detaylar: suspicious_labels.csv")

    # Görsel inceleme — ilk 6 şüpheli
    n_show = min(6, len(suspicious_labels))
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    axes = axes.flatten()
    for j in range(n_show):
        item = suspicious_labels[j]
        img_bgr = cv2.imread(item["path"])
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        axes[j].imshow(img_rgb)
        axes[j].set_title(
            f"GT:{item['true_label']} Pred:{item['pred_label']} ({item['confidence']:.0%})",
            fontsize=9,
        )
        axes[j].axis("off")
    for j in range(n_show, len(axes)):
        axes[j].axis("off")
    plt.suptitle("Olası Etiket Hataları — Manuel Doğrulama Gerekli")
    plt.tight_layout()
    plt.savefig("label_error_review.png", dpi=150)
    plt.show()
else:
    print("Yüksek güven eşiğinde şüpheli etiket bulunamadı.")


In [ ]:
# Inference için en iyi sınıflandırıcıyı yükle
model_cls, _ = create_model(BEST_ARCHITECTURE, num_classes=len(CLASS_NAMES))
model_cls.load_state_dict(torch.load(best_model_path, map_location=device))
model_cls = model_cls.to(device)
model_cls.eval()
inference_transform = eval_transform

print(f"En iyi model: {BEST_ARCHITECTURE}")
print(f"Ağırlıklar: {best_model_path.resolve()}")
print(f"K-Fold F1: {cv_comparison_df.loc[cv_comparison_df['Model']==BEST_ARCHITECTURE, 'F1'].values[0]:.4f}")


In [ ]:
# Entegre Pipeline: YOLO11 + En İyi Sınıflandırıcı + Hibrit KDS

def classify_ship_crop(crop_bgr, cls_model, transform, device):
    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    crop_pil = Image.fromarray(crop_rgb)
    tensor = transform(crop_pil).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = cls_model(tensor)
        probs = torch.softmax(logits, dim=1)[0]
        pred_idx = logits.argmax(dim=1).item()
    fuel_prob = probs[0].item()
    other_prob = probs[1].item()
    return CLASS_NAMES[pred_idx], fuel_prob, other_prob


def draw_decision_panel(canvas, report, x1, y1, x2, y2):
    is_fuel = report["Fuel / Other"].lower() == "fuel"
    color = (0, 0, 255) if is_fuel else (0, 200, 0)
    cv2.rectangle(canvas, (x1, y1), (x2, y2), color, 2)

    lines = [
        f"Ship Type       : {report['Ship Type'][:36]}",
        f"Fuel/Other      : {report['Fuel / Other']}",
        f"Tanker Category : {report['Tanker Category']}",
        f"Risk            : {report['Risk Score']}/100 ({report['Risk Level']})",
        f"Env. Threat     : {report['Environmental Risk Score']}/100",
        f"Strategic Score : {report['Strategic Importance Score']}/100",
        f"Confidence      : {report['Confidence Score']}/100",
        f"Route           : {report['Estimated Route']}",
    ]
    h_img, w_img = canvas.shape[:2]
    panel_h = len(lines) * 22 + 16
    panel_w = 430
    py = y2 + 10 if y2 + panel_h + 10 < h_img else max(5, y1 - panel_h - 10)
    px = max(5, min(x1, w_img - panel_w - 5))
    overlay = canvas.copy()
    cv2.rectangle(overlay, (px, py), (px + panel_w, py + panel_h), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.65, canvas, 0.35, 0, canvas)
    for i, line in enumerate(lines):
        cv2.putText(canvas, line, (px + 8, py + 18 + i * 22),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1, cv2.LINE_AA)
    return canvas


def run_integrated_analysis(img_path, yolo_model, cls_model, transform, device,
                            conf=0.35, iou=0.45, save_path="decision_support_result.jpg"):
    img = cv2.imread(str(img_path))
    if img is None:
        raise FileNotFoundError(f"Görüntü okunamadı: {img_path}")
    img_h, img_w = img.shape[:2]
    canvas = img.copy()
    results = yolo_model.predict(source=str(img_path), conf=conf, iou=iou, verbose=False)
    reports, fuel_reports = [], []

    if len(results[0].boxes) == 0:
        print("Görselde gemi tespit edilemedi.")
        return canvas, reports, fuel_reports

    for idx, box in enumerate(results[0].boxes, start=1):
        det_conf = float(box.conf[0])
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(img_w, x2), min(img_h, y2)
        bw, bh = x2 - x1, y2 - y1
        if bw <= 0 or bh <= 0:
            continue
        crop = img[y1:y2, x1:x2]
        if crop.size == 0:
            continue

        fuel_label, fuel_prob, other_prob = classify_ship_crop(crop, cls_model, transform, device)
        report = build_decision_support_report(
            fuel_label, bw, bh, img_w, img_h, det_conf, fuel_prob, other_prob
        )
        reports.append(report)
        print_decision_support_report(report, ship_id=idx)
        if fuel_label == "fuel":
            fuel_reports.append(report)
            txt = generate_fuel_analysis_report(report, ship_id=idx)
            with open(f"fuel_report_ship_{idx}.txt", "w", encoding="utf-8") as f:
                f.write(txt)
        canvas = draw_decision_panel(canvas, report, x1, y1, x2, y2)

    cv2.imwrite(save_path, canvas)
    print(f"\nSonuç: {save_path} | Fuel raporları: fuel_report_ship_*.txt")
    return canvas, reports, fuel_reports


demo_candidates = []
if "base_path" in dir() and os.path.exists(base_path):
    p = os.path.join(base_path, "EOY_029.jpg")
    if os.path.exists(p):
        demo_candidates.append(p)
fuel_samples = sorted((DATASET_ROOT / "fuel").glob("*.jpg"))
if fuel_samples:
    demo_candidates.append(str(fuel_samples[0]))

demo_img_path = demo_candidates[0]
print(f"Demo: {demo_img_path}")
canvas, all_reports, fuel_reports = run_integrated_analysis(
    demo_img_path, model, model_cls, inference_transform, device,
)
plt.figure(figsize=(14, 10))
plt.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title(f"YOLO11 + {BEST_ARCHITECTURE} + Hibrit KDS")
plt.savefig("decision_support_visualization.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# =============================================================================
# Bilimsel Katkılar ve Yenilikler (Bitirme Projesi / Akademik Yayın)
# =============================================================================

contributions = [
    "1. Hibrit Karar Destek Sistemi (HKDS): Kural tabanlı yaklaşım yerine çoklu özellik",
    "   (bbox alanı, en-boy oranı, YOLO güveni, Fuel olasılığı, görüntü çözünürlüğü,",
    "   gemi yönü) ile skor tabanlı tanker kategorisi tahmini.",
    "2. Sayısal Risk Modellemesi: Çevresel tehdit (0–100), stratejik önem (0–100) ve",
    "   birleşik risk puanı ile Düşük/Orta/Yüksek/Kritik sınıflandırması.",
    "3. İki Aşamalı Derin Öğrenme Pipeline: YOLO11 (tespit) + ResNet18/EfficientNet-B0",
    "   (sınıflandırma) entegrasyonu yüksek çözünürlüklü uydu görüntülerinde.",
    "4. K-Fold Cross Validation: Küçük veri setlerinde güvenilir performans tahmini;",
    "   ResNet18 vs EfficientNet-B0 karşılaştırmalı analiz.",
    "5. Grad-CAM ile Model Yorumlanabilirliği: Sınıflandırıcı kararlarının görsel",
    "   açıklanabilirliği — akademik yayın için kritik katkı.",
    "6. Otomatik Etiket Hatası Tarama: Yüksek güvenli yanlış tahminlerin tespiti ile",
    "   veri kalitesi iyileştirme metodolojisi.",
    "7. Otomatik Fuel Tanker Analiz Raporu: Her fuel gemisi için yapılandırılmış",
    "   risk ve stratejik değerlendirme raporu üretimi.",
    "8. YOLO11 Hiperparametre Optimizasyonu: VHR uydu verisi için önerilen epoch,",
    "   batch, imgsz ve learning rate konfigürasyonu.",
]

print("BİLİMSEL KATKILAR VE YENİLİKLER")
print("=" * 70)
for line in contributions:
    print(line)

print("\n=== ÖZET METRİK TABLOSU ===")
summary = pd.merge(cv_comparison_df, test_comparison_df, on="Model", suffixes=("_cv", "_test"))
display(summary.round(4))
summary.to_csv("full_model_comparison.csv", index=False)
